# Command center

The workspace for this project: every knob, the setup that acts on them, the features being
built, and the output you check them by. Runs top to bottom on a fresh runtime — no hidden
state, no cell that has to run out of order.

It runs on **two hosts**. On **Kaggle** the competition data is mounted and the pixels are
local; that is where anything touching DICOM belongs, because the dataset is ~570 GB. On
**Colab** the setup cell mounts Drive and clones the repo; that is where iteration on a cached
subset is comfortable. The config cell detects which one it is on and resolves paths
accordingly — nothing below has to know.

Config comes **first**, before anything acts: read one cell and you know everything this
project can be told to do. Setup then consumes those values. Nothing below reads a setting
from anywhere else.

Features get built *here*, in as many cells as it takes to see them working, and move into
`src/` once they've stopped changing. `AGENTS.md` §6 has that lifecycle; `STRATEGY.md` has why
each feature is worth building; `PLAYBOOK.md` has the manual setup and the environment quirks.

**One standing rule:** competition data — report text, pixel data, per-study rows — never leaves
this notebook for an agent session. Print aggregates, not records. `PLAYBOOK.md` § Gotchas.


## 1. Config

Every knob, in one cell, before anything runs. Edit here, then run everything below.

In [ ]:
"""Every knob this project has, and why each is set the way it is.

Read this cell on its own and you should know what you can change, what the alternatives were,
and why the current value won — that last part is what gets forgotten first. Grouped by the
decision you're making, not by whichever module consumes the value.

This cell only *declares*. It mounts nothing, creates nothing, downloads nothing — so you can
re-read and re-run it any time without side effects. Setup, below, is what acts on it.
"""

from pathlib import Path

# --- Which host is this? ------------------------------------------------------------------
# The same notebook runs on Kaggle (competition data mounted, no Drive) and on Colab (Drive
# mounted, no competition data). Everything below that differs between them is derived from
# this one flag, so there is exactly one branch in the whole notebook.
#
# Detection — alternatives considered:
#   /kaggle/input exists          <- current. present on every Kaggle kernel, absent elsewhere,
#                                    and true before any import has happened
#   importing google.colab           tells you it is Colab, but by raising ImportError somewhere
#                                    else, which is a worse place to find out
#   an environment variable          would have to be set by hand on one of the two hosts, which
#                                    is exactly the manual step this avoids
ON_KAGGLE = Path("/kaggle/input").exists()

# --- The competition ----------------------------------------------------------------------
# The slug is the folder name the data mounts under on Kaggle, so it is load-bearing rather
# than decorative. Setup uses it to check the dataset is actually attached before anything
# below assumes it.
COMPETITION = "rsna-knee-abnormality-detection"

# --- Where files live ---------------------------------------------------------------------
# Everything that isn't code, config or docs: data, weights, caches, outputs, secrets.
# None of it belongs in git — .gitignore keeps it out.
#
# DRIVE_ROOT (Colab only) — alternatives considered:
#   MyDrive/projects/<project>  <- current. survives runtime teardown, and the shared parent
#                                  means MyDrive's root gains one folder however many projects
#                                  you start, rather than one folder per project
#   MyDrive/<project>              a click shallower. clutters the root once there are several
#   /content/<anything>            the runtime's own disk. faster, but wiped whenever the
#                                  runtime recycles, which eventually means lost work
#
# On Kaggle there is no equivalent choice: /kaggle/working is the only writable place that
# becomes an attachable output for the next kernel, which is how the pipeline passes large
# artifacts along without anything crossing a local machine.
DRIVE_ROOT = Path("/content/drive/MyDrive/projects/rsna-claude")
KAGGLE_ROOT = Path("/kaggle/working")

WORK_ROOT = KAGGLE_ROOT if ON_KAGGLE else DRIVE_ROOT
DATA_DIR = WORK_ROOT / "data"           # derived data we produce: caches, labels, folds
OUTPUTS_DIR = WORK_ROOT / "outputs"     # anything a run produces: predictions, metrics, models
SECRETS_DIR = DRIVE_ROOT / "secrets"    # Colab fallback for kaggle.json (PLAYBOOK.md); prefer
                                        # Colab Secrets, which keeps nothing on disk

# The competition data itself, read-only, and only on Kaggle. Never copied, never mirrored to
# Drive — at ~570 GB it does not fit anywhere else, which is the whole reason Kaggle is the
# primary host.
COMPETITION_DIR = Path("/kaggle/input") / COMPETITION

# --- Where the code comes from ------------------------------------------------------------
# Opening this notebook gives you the notebook and nothing else, so the repo is fetched
# separately to put `src/` on the import path. That's what makes `from pipeline import ...`
# work further down, on either host.
#
# REPO_DIR — alternatives considered:
#   /content/repo | /kaggle/working/repo  <- current. fast, and a clone is disposable
#   under Drive                              would survive restarts, but git over Drive's FUSE
#                                            mount is slow and occasionally corrupts .git
#
# REPO_BRANCH — which branch the runtime imports from:
#   main              <- current. what has landed, and what you normally want
#   a feature branch     to check work before it is merged. Open the notebook from that branch
#                        too — the README badge always points at main, so change `blob/main` to
#                        `blob/<branch>` in the URL. Set it back to main afterwards.
REPO_URL = "https://github.com/vak-sah/rsna-claude.git"
REPO_BRANCH = "main"
REPO_DIR = (KAGGLE_ROOT if ON_KAGGLE else Path("/content")) / "repo"

# --- Dependencies -------------------------------------------------------------------------
# Packages the host doesn't already ship. Pin versions — a runtime six months from now should
# resolve to what this one did. Kept empty as long as possible: on Kaggle a pip install inside
# a *submission* notebook is minutes of pure runtime loss, and at 72 seconds per 0.001 AUC in
# the efficiency track (STRATEGY.md §3) that is expensive. Prefer what is preinstalled.
PACKAGES: list[str] = [
    # "some-lib==1.2.3",
]

# --- Project knobs ------------------------------------------------------------------------
# Empty until this project has features of its own. Each arrives as its own block: the value,
# the alternatives weighed, and one line on why the current value won. STATE.md § Next is the
# order they are expected to arrive in.

print(f"host          {'Kaggle' if ON_KAGGLE else 'Colab'}")
print(f"competition   {COMPETITION}")
print(f"WORK_ROOT     {WORK_ROOT}")
print(f"DATA_DIR      {DATA_DIR}")
print(f"OUTPUTS_DIR   {OUTPUTS_DIR}")
print(f"REPO_URL      {REPO_URL} ({REPO_BRANCH})")
print(f"PACKAGES      {PACKAGES or 'none'}")


## 2. Setup

Acts on the config above. Run once per fresh runtime; re-run after **Runtime > Restart**.

In [ ]:
"""Make the runtime match the config above: mount Drive if needed, fetch the repo, install.

There are no settings in this cell by design — every value it uses comes from the config cell,
so there is exactly one place to look when changing how a run behaves. If you find yourself
about to hardcode something here, it belongs up there instead.

Idempotent, and safe to re-run after changing REPO_BRANCH: an existing clone is moved onto the
requested branch rather than re-cloned, and the folders are created only if missing.

What you should see: the host, a confirmation that the competition data is attached (on Kaggle)
or that Drive is mounted (on Colab), and the branch the clone is on. A warning about the
competition data means the dataset is not attached to this kernel — fix that before running
anything below, or it will fail in a much less obvious way.
"""

import subprocess
import sys

if ON_KAGGLE:
    if not COMPETITION_DIR.exists():
        print(
            f"\n!!  {COMPETITION_DIR} is not there.\n"
            "    Attach the competition dataset to this kernel (Add Input -> Competitions),\n"
            "    and accept the competition rules first if you have not.\n"
        )
else:
    from google.colab import drive

    drive.mount("/content/drive")

for _d in (DATA_DIR, OUTPUTS_DIR):
    _d.mkdir(parents=True, exist_ok=True)

if (REPO_DIR / ".git").exists():
    # fetch + checkout + reset rather than a bare `pull`: pull would update whatever branch
    # happened to be checked out, so changing REPO_BRANCH between runs would silently do
    # nothing. The clone is disposable and nobody edits it, so a hard reset is safe and makes
    # the runtime match the branch exactly.
    _git = ["git", "-C", str(REPO_DIR)]
    subprocess.run([*_git, "fetch", "--quiet", "origin", REPO_BRANCH], check=True)
    subprocess.run([*_git, "checkout", "--quiet", REPO_BRANCH], check=True)
    subprocess.run([*_git, "reset", "--hard", "-q", f"origin/{REPO_BRANCH}"], check=True)
else:
    REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        ["git", "clone", "--quiet", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)],
        check=True,
    )

if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))

if PACKAGES:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *PACKAGES], check=True)

_where = "competition data attached" if ON_KAGGLE and COMPETITION_DIR.exists() else (
    "competition data MISSING" if ON_KAGGLE else "Drive mounted"
)
print(f"{'Kaggle' if ON_KAGGLE else 'Colab'} \u00b7 {_where} \u00b7 {REPO_DIR} on {REPO_BRANCH} "
      f"\u00b7 {len(PACKAGES)} extra package(s)")


## 3. Pipeline

The end-to-end stub: read a file, pass the records through untouched, write them out. It proves
notebook -> `src/` -> `tests/` -> CI is connected, and gives the first real feature something to
replace. `src/pipeline/stub.py` owns it — delete it once a real stage exists.

In [ ]:
"""Run the stub end to end and show the result.

Writes a small input file into DATA_DIR, runs the pipeline, prints what came back out. Replace
this call when the stub is replaced by a real first stage.
"""

from pipeline import stub

in_path = DATA_DIR / "stub_input.txt"
out_path = OUTPUTS_DIR / "stub_output.txt"

in_path.write_text("alpha\nbeta\ngamma\n")

written = stub.run(in_path, out_path)

print(f"{written} record(s) -> {out_path}\n")
print(out_path.read_text())